# AliasingAtlas — sampling, aliasing & reconstruction, hands-on

<a href="https://colab.research.google.com/github/Boussetta/NyquistNavigator/blob/main/AliasingAtlas.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Move a slider and see what happens to a signal when it is **sampled**, **quantized**, and **reconstructed**. This is the Nyquist–Shannon sampling theorem as an interactive lab.

**How to use this notebook**
1. Run the **Setup** cell. In Colab, the first run takes about 30 s while it installs the package.
2. Run the **Launch** cell. The interactive figure appears, and audio players and download links show up just below it.
3. Work through the **Guided experiments**. Each one takes about 2 minutes.

> Colab may warn that this notebook was not authored by Google. It is fetched from the public GitHub repository; choose **Run anyway**.


## 1 · Setup

In [ ]:
# Setup: installs AliasingAtlas in Colab, or uses the local checkout.
import importlib.util
import subprocess
import sys
from pathlib import Path

REPO = "Boussetta/NyquistNavigator"
REF = "main"  # pin a tag (e.g. "v1.1.0") for a reproducible classroom setup


def in_colab() -> bool:
    try:
        import google.colab  # type: ignore[import-not-found]  # noqa: F401
    except ImportError:
        return False
    return True


def find_project_root(start: Path):
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists() and (path / "src" / "aliasing_atlas").exists():
            return path
    return None


def pip_install(*packages: str) -> None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)


is_colab = in_colab()

if is_colab:
    print("Installing AliasingAtlas and the interactive backend (about 30 s on first run)...")
    pip_install("ipympl", f"aliasing-atlas @ git+https://github.com/{REPO}.git@{REF}")
    from google.colab import output  # type: ignore[import-not-found]

    output.enable_custom_widget_manager()
elif importlib.util.find_spec("aliasing_atlas") is None:
    # Local checkout: import straight from src/ without installing.
    root = find_project_root(Path.cwd())
    if root is None:
        raise ModuleNotFoundError(
            "aliasing_atlas not found. Run `pip install -e \".[notebook]\"` from the project root, "
            "or open this notebook from inside the repository."
        )
    sys.path.insert(0, str(root / "src"))

try:
    get_ipython().run_line_magic("matplotlib", "widget")
    INTERACTIVE = True
except Exception as exc:  # ipympl missing: fall back to static images
    get_ipython().run_line_magic("matplotlib", "inline")
    INTERACTIVE = False
    print(f"WARNING: interactive backend unavailable ({exc}). Sliders will not respond.")
    print("Fix: pip install ipympl, then restart the kernel and run all cells.")

import aliasing_atlas
from aliasing_atlas.app import AliasingToolbox

print(f"Ready: AliasingAtlas {aliasing_atlas.__version__} | interactive={INTERACTIVE} | colab={is_colab}")


## 2 · Launch

**Reading the screen**

| Panel | What it shows |
|---|---|
| **Time Domain** (top left) | The ideal signal (dashed), the samples (dots), and the reconstruction (green FFT, or cyan FOH line). |
| **Magnitude Spectrum** (top right) | What the *sampled* signal contains, from 0 to fs/2. Orange = Nyquist limit, blue = highest real component, magenta = where an alias will appear. |
| **Phase Spectrum** (bottom left) | Phase of each frequency bin. |
| **Quantization Error** (bottom right) | Rounding error added by the converter's finite bit depth. |

The two big sliders set the **signal frequency** and the **sampling rate**. The **Sections** buttons switch the lower panel between *Signal*, *Sampling* (bits, window, anti-alias filter, reconstruction) and *Audio* (play and export). **Presets** load ready-made scenarios, and **Learn → On** shows a hint at the top. The status bar reports the reconstruction error (RMSE, SNR) and turns orange when aliasing occurs.


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display

try:
    import ipywidgets as widgets

    panel = widgets.Output()  # audio players and download links appear here
except ImportError:
    widgets, panel = None, None

with plt.ioff():
    toolbox = AliasingToolbox(output=panel)

if INTERACTIVE and widgets is not None:
    display(widgets.VBox([toolbox.fig.canvas, panel]))
else:
    toolbox.show()


## 3 · Guided experiments

Leave the figure above open and scroll back up to it for each experiment. Press **Reset All** between experiments.

### Experiment 1: Watch aliasing happen
1. Click the **Aliasing** preset: a 10 Hz sine sampled at 15 Hz.
2. The dots no longer follow the dashed 10 Hz wave. They trace a slower **5 Hz** sine, and the spectrum peak lands on the magenta *Predicted Alias* marker at |10 − 15| = 5 Hz.
3. Slowly drag **Sampling** up. The alias marker slides toward 10 Hz, and at **fs = 20 Hz = 2·f** the warning disappears.

*Takeaway:* a sampled signal cannot tell a 10 Hz tone from a 5 Hz one when fs = 15 Hz. Any frequency above fs/2 folds back to |f − k·fs|.

### Experiment 2: Right at the Nyquist limit
1. Click **Near Nyquist** (10 Hz at fs = 22 Hz, barely 2.2 samples per cycle). The samples look jagged, yet FFT reconstruction is essentially perfect (SNR ≈ 97 dB).
2. Open **Sampling → Recon → FOH** (straight lines between samples). SNR drops to about 20 dB.

*Takeaway:* Nyquist says the information is **all there**. Getting it back requires ideal band-limited (sinc/FFT) interpolation, not connect-the-dots.

### Experiment 3: Why anti-alias filters exist
1. **Reset All**, choose waveform **Square**, and set **Fourier Harm.** to 15. Its harmonics reach 290 Hz, far above fs/2 = 50 Hz, so the status turns orange.
2. In **Sampling → AAF**, pick **Ideal**. The filter removes everything above 50 Hz *before* sampling: nothing can fold back, but the square edges are gone (that detail is lost).
3. Try **Butter**, a realistic analog filter. It attenuates instead of removing, so a little aliasing remains.

### Experiment 4: Bits and the "6 dB per bit" rule
1. **Reset All**, then lower **Sampling → Bit Depth** from 16 to 8, 4, and 3.
2. Watch SNR: about 97 → 47 → 24 → 16 dB. Each bit removed costs about 6 dB, close to the textbook 6.02·N + 1.76 dB. The quantization error plot shows the staircase noise growing.

### Experiment 5: AM sidebands
1. Click **AM Sidebands**: a 25 Hz carrier modulated by 3 Hz at depth m = 0.7.
2. The spectrum shows the carrier (1.0) and two sidebands at 22 and 28 Hz with amplitude m/2 = 0.35. Switch **Scale → dB Scale** to see the floor.
3. Raise **Message Freq** and watch the sidebands move apart and the required sampling rate (*Nyquist ≥* in the top line) grow.

### Experiment 6: Spectral leakage and windows
1. **Reset All**, then set the base frequency to about **10.3 Hz**. The window no longer holds a whole number of cycles, so the peak smears across neighbouring bins (leakage), and the FFT reconstruction shows small errors at the window edges.
2. Choose **Scale → dB Scale**, then compare **Window → None / Hann / Hamming**. Windows trade a wider peak for much lower side-lobes. The window affects only the spectrum, never the reconstruction.

### Experiment 7: Listen and export
Open **Sections → Audio**. **Play Audio** shows a player below the figure, where *Sampled* is the stair-step signal and *Recon* the reconstruction. **Export CFG** and **Export WAV** save files and show a download link (in Colab, files are also under `/content/exports`).


## 4 · Use the DSP functions directly (optional)

Every curve in the GUI comes from plain functions in `aliasing_atlas.dsp`, so you can script your own experiments. This cell recreates Experiment 1 as a static figure.


In [ ]:
import numpy as np
from aliasing_atlas import dsp

f_sig, f_samp = 10.0, 15.0
alias = dsp.folded_alias_frequency(f_sig, f_samp)

t, n_samples, step = dsp.build_sampling_grid(f_samp, duration=1.6, min_points=2000)
y = np.sin(2 * np.pi * f_sig * t)
t_s, y_s = t[::step], y[::step]
y_rec = dsp.reconstruct_fft(y_s, len(t))

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(t, y, "--", alpha=0.5, label=f"true {f_sig:g} Hz")
ax.plot(t, y_rec, lw=2, label=f"what the samples say ({alias:g} Hz)")
ax.plot(t_s, y_s, "o", color="darkred", label=f"samples at {f_samp:g} Hz")
ax.set(xlabel="Time (s)", title=f"Aliasing: {f_sig:g} Hz sampled at {f_samp:g} Hz looks like {alias:g} Hz")
ax.legend(loc="upper right", fontsize=8)
plt.show()


## Troubleshooting

- **Sliders don't respond / the figure is a static image.** The interactive backend didn't load. Use *Runtime → Restart session*, then *Runtime → Run all*.
- **Moving a slider has no effect after a restart.** Run the Setup and Launch cells again; widgets don't survive a kernel restart.
- **No sound.** Click ▶ on the player below the figure (browsers block autoplay) and check your volume. Signal frequencies top out at 50 Hz, so small laptop speakers may barely reproduce them; headphones help.
- **Where are my exports?** Under `exports/` next to the notebook (`/content/exports` in Colab), plus the download link below the figure.

Source code, issues, and documentation: [github.com/Boussetta/NyquistNavigator](https://github.com/Boussetta/NyquistNavigator)
